# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook provides a step-by-step guide to loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. Following the Croissant metadata standard, all dataset entities are referenced via their `@id` fields.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# If necessary, install the mlcroissant library
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and prepare for record set exploration using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata for inspection
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("")
print("Description:")
print(metadata.description)


## 2. Data Overview

Review `recordSet` entities, fields, and column `@id`s as described in the Croissant package. For all operations, we will reference entities by their `@id` values.

In [ ]:
# Let's examine all record sets available in the dataset.
record_sets = list(dataset.record_sets)

print("Available Record Sets and fields:")
overview_info = []
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    field_ids = [field.id for field in rs.fields]
    print(f"  Fields (@id): {field_ids}")
    overview_info.append({
        "name": rs.name, 
        "@id": rs.id, 
        "fields": field_ids
    })
    print("")

# Preview a couple of rows from the first record set for reference
if record_sets:
    sample_rs_id = record_sets[0].id
    print(f"Sample records from RecordSet (@id): {sample_rs_id}\n")
    for i, x in enumerate(dataset.records(record_set=sample_rs_id)):
        pprint.pprint(x)
        if i >= 2:
            break
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction

Extract data from each record set into a pandas DataFrame using their `@id`. We'll build a dictionary mapping record set `@id`s to DataFrames.


In [ ]:
# Extract all data into DataFrames, using record set @id as key
dataframes = {}
for rs in dataset.record_sets:
    rs_id = rs.id
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from RecordSet @id: {rs_id}")
    if len(df) > 0:
        print(f"  Columns (@id): {df.columns.tolist()}")
    print("")

# For further steps, select the first record set with data
selected_record_set_id = None
for rs_id, df in dataframes.items():
    if len(df) > 0:
        selected_record_set_id = rs_id
        break
if selected_record_set_id is not None:
    print(f"Example DataFrame columns for {selected_record_set_id}:\n{dataframes[selected_record_set_id].columns.tolist()}")
    display(dataframes[selected_record_set_id].head())
else:
    print("No non-empty record sets were found.")

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate filtering, normalization, and grouped analysis steps, referencing columns by their `@id`. Update the selected field `@id`s based on results from the data overview above.

In [ ]:
# If there is a record set loaded, proceed with simple EDA
from pandas.api.types import is_numeric_dtype

if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    
    # Choose a numeric field by @id from the DataFrame
    numeric_field_id = None
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id:
        print(f"Using numeric field (@id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().sum() > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Records with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)}")
        display(filtered_df.head())

        # Normalize the field
        colnorm = f"{numeric_field_id}_normalized"
        filtered_df[colnorm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} (z-score) for filtered records:")
        display(filtered_df[[numeric_field_id, colnorm]].head())

        # Try grouping by a non-numeric field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id:
            print(f"Grouping by field (@id): {group_field_id}")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            display(grouped.head())
        else:
            print("No suitable group field (categorical) found in this record set.")
    else:
        print("No numeric fields found in this record set for EDA.")
else:
    print("No data available to analyze.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its normalization if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and numeric_field_id and len(filtered_df) > 0:
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)

    plt.subplot(1, 2, 2)
    sns.histplot(filtered_df[colnorm], kde=True, color='orange')
    plt.title(f"Normalized {numeric_field_id}")
    plt.xlabel(colnorm)

    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion

- This notebook demonstrated how to load, explore, and analyze a FAIR^2 Croissant dataset using `mlcroissant` while referencing all recordset elements by their `@id` fields.
- For further analysis, inspect all fields, column `@id`s, and utilize the rich schema to perform more domain-specific exploration, statistics, and modeling.
- For questions on the dataset schema, see metadata at the [FAIR^2 schema link](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).